## Utilidades

In [2]:
import pandas as pd


In [3]:
contracts = pd.read_csv("contracts.csv")
customers = pd.read_csv("customers.csv")
usage     = pd.read_csv("usage.csv")

# 1. Descubrimiento y Comprensión de Datos

In [63]:
#contracts.head()
customers.head(-10) 
#usage.head()

,customer_id,name,email,registration_date,category,internal_score
0,C_00001,User_1,user_1@domain.com,2022-08-02,A,41.05
1,C_00002,User_2,user_2@domain.com,2019-08-17,X,14.15
2,C_00003,User_3,user_3@domain.com,2019-02-21,B,43.82
3,C_00004,User_4,user_4@domain.com,2023-02-27,A,63.70
4,C_00005,User_5,user_5@domain.com,2020-07-17,A,58.06
...,...,...,...,...,...,...
4985,C_04986,User_4986,user_4986@domain.com,2019-12-23,A,45.50
4986,C_04987,User_4987,user_4987@domain.com,2019-03-05,B,34.93
4987,C_04988,User_4988,user_4988@domain.com,2021-04-08,B,51.90
4988,C_04989,User_4989,user_4989@domain.com,2023-10-17,B,44.50


### Renombrar columnas

In [ ]:
contracts.rename(columns={
    "con_id":   "contract_id", # PK unica = 5500 registros
    "cli_ref":  "customer_id", # FK a customers (3395 coincidencias, 2 se repiten)
    "p_type":   "plan_type", # tipo de plan 
    "val_mo":   "monthly_value", # valor mensual del contrato
    "st":       "status", # estado del contrato (3 valores: )
    "dt_start": "start_date", # fecha de inicio del contrato
}, inplace=True)

customers.rename(columns={
    "id_cli":       "customer_id", # PK unica (revisar, no es unica) = 4994 registros // solución: coincidir entre customer_id y name
    "full_name":    "name", # nombre del cliente
    "contact_info": "email", # Contacto
    "reg_date":     "registration_date", # fecha de registro
    "cat":          "category", # 4 valores : A, B, C, X
    "int_score":    "internal_score", # puntaje interno del cliente (0-100)
}, inplace=True)

usage.rename(columns={
    "u_id":  "usage_id", # PK unica
    "c_ref": "contract_id", # FK contracts
    "qty":   "quantity", # consumo
    "ts":    "timestamp", # fecha y hora del consumo
    "loc":   "location", # ubicacion del consumo
}, inplace=True)

### Exploración

In [37]:
customer_id_nunique = customers['customer_id'].nunique()
contracts_id_nunique = contracts['contract_id'].nunique()
customer_id_nunique

4994

In [9]:
customer_id2_nunique = customers['customer_id'].nunique()
customer_id2_nunique

4994

In [32]:
# Cuenta cuántos registros de contracts tienen un customer_id presente en customers
coincidencias = contracts['customer_id'].isin(customers['customer_id']).sum()
coincidencias

np.int64(5497)

In [ ]:
# Valores únicos comunes
comunes = len(set(contracts['customer_id']) & set(customers['customer_id']))
comunes  # 3393 coincidencias y 2 se repiten esto es porque un cliente puede tener más de un contrato, pero un contrato no puede tener más de un cliente. (1:N)

3393

In [ ]:
filas_duplicadas = customers[customers['customer_id'].duplicated(keep=False)].sort_values(by='registration_date')
filas_duplicadas 

,customer_id,name,email,registration_date,category,internal_score
4994,C_00001,User_4995,user_4995@domain.com,2019-07-13,A,35.17
4993,C_00001,User_4994,user_4994@domain.com,2020-05-26,B,49.27
4995,C_00001,User_4996,user_4996@domain.com,2021-08-04,C,83.61
4992,C_00001,User_4993,user_4993@domain.com,2022-01-31,A,59.40
4990,C_00001,User_4991,user_4991@domain.com,2022-02-12,X,50.34
0,C_00001,User_1,user_1@domain.com,2022-08-02,A,41.05
4991,C_00001,User_4992,user_4992@domain.com,2023-03-13,A,35.20


In [48]:
contract_C_01 = contracts[contracts['customer_id'] == 'C_00001']
contract_C_01

,contract_id,customer_id,plan_type,monthly_value,status,start_date
1838,CON_001839,C_00001,M2,112.97,-1,2020-11-17


In [59]:
contract_U_01 = usage[usage['contract_id'] == 'CON_001839']
contract_U_01

,usage_id,contract_id,quantity,timestamp,location
8327,U_0008328,CON_001839,15.72,2024-03-12 15:56:16,US
9048,U_0009049,CON_001839,1.29,2024-06-08 04:04:20,UK
10047,U_0010048,CON_001839,0.42,2024-10-20 13:09:40,US
18951,U_0018952,CON_001839,34.59,2024-03-06 22:00:13,ES
20086,U_0020087,CON_001839,2.83,2024-07-05 23:20:39,??
30360,U_0030361,CON_001839,11.77,2024-01-20 18:48:47,DE
31936,U_0031937,CON_001839,1.71,2024-12-02 02:16:53,ES
33495,U_0033496,CON_001839,8.97,2024-10-13 01:13:04,??
42835,U_0042836,CON_001839,2.00,2024-11-25 20:34:00,DE
46541,U_0046542,CON_001839,14.93,2024-04-17 03:23:21,FR


In [66]:
# Teoria: Si el formato de customer_id es consistente, debería ser algo como 'C_00001', 'C_00002', etc. Esto implica que el número después de 'C_' 
# debería coincidir entre contracts y customers para el mismo cliente.
# Uso .extract para string con regex hecha con IA (busca 1 o más dígitos) y luego convierto a int para comparar numéricamente.
id_num = customers['customer_id'].str.extract(r'(\d+)').astype(int)
name_num = customers['name'].str.extract(r'(\d+)').astype(int)

# Filtrar las filas donde NO coinciden (patrón se rompe)
no_coinciden = customers[id_num[0] != name_num[0]]
no_coinciden

# Entiendo que cada customer_id tiene coincidencia por name y en este caso de trata de 6 clientes que tenian mal asignado el custormer_id ("conclusión")

,customer_id,name,email,registration_date,category,internal_score
4990,C_00001,User_4991,user_4991@domain.com,2022-02-12,X,50.34
4991,C_00001,User_4992,user_4992@domain.com,2023-03-13,A,35.20
4992,C_00001,User_4993,user_4993@domain.com,2022-01-31,A,59.40
4993,C_00001,User_4994,user_4994@domain.com,2020-05-26,B,49.27
4994,C_00001,User_4995,user_4995@domain.com,2019-07-13,A,35.17
4995,C_00001,User_4996,user_4996@domain.com,2021-08-04,C,83.61
